<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/Dataset_Manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Wearable Biomechanics - Dataset Manager**

To manage the datasets in Google Drive and to easily access then from Colab Notebook we use [OpenDatasetStore](https://github.com/janithcyapa/OpenDatasetStore) Python utility

## **🛠️ Setup OpenDatasetStore**

In [6]:

from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!wget -q -O open_dataset_store.py https://raw.githubusercontent.com/janithcyapa/OpenDatasetStore/refs/heads/main/open_dataset_store.py

from open_dataset_store import OpenDatasetStore

# @markdown ### Store Configuration
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}

store = OpenDatasetStore(
    base_dir=dataset_path,
    entity_id_format=f"sub_{{num:04d}}",
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)

Mounted at /content/drive
Store initialised at: /content/drive/MyDrive/Datasets/Wearable-Biomechanics


# **👤 People Management (Subject)**

In [8]:
# @title 📝 Register New Subject { display-mode: "form" }
# @markdown Enter the details of the test subject to generate a unique ID.

name = "Hivindu Kirulapana" # @param {type:"string"}
age = 22 # @param {type:"number"}
weight_kg = 72 # @param {type:"number"}
height_m = 1.74 # @param {type:"number"}
experience_years = 1 # @param {type:"integer"}
notes = "Participation in Other Sports - Rugby" # @param {type:"string"}
achievement = ""# @param {type:"string"}
best_weight = ""# @param {type:"string"}

# Logic to create entity
sub_id = store.create_entity(
    entity_type="subjects",
    name=name,
    age=age,
    weight=weight_kg,
    height=height_m,
    experience=experience_years,
    achievement=achievement,
    best_weight=best_weight,
    notes=notes
)

print(f"✅ Created Subject ID: {sub_id}")

Entity 'sub_0001' created in subjects.
✅ Created Subject ID: sub_0001


In [9]:
# @title 📋 Participant Registry { display-mode: "form" }
# @markdown View all registered subjects in the database.

df_subjects = store.list_entities("subjects")

if not df_subjects.empty:
    # Use Colab's interactive data table
    from google.colab import data_table
    display(data_table.DataTable(df_subjects, include_index=False, num_rows_per_page=10))
else:
    print("Empty registry.")

,entity_id,name,age,weight,height,experience,achievement,best_weight,notes
0,sub_0001,Hivindu Kirulapana,22,72,1.74,1,,,Participation in Other Sports - Rugby


# **📂 Experiment Upload & Data Exploration**

In [ ]:
# @title 🚀 Upload New Experiment { display-mode: "form" }
# @markdown Select a subject and upload their raw IMU data.

target_subject_id = "subj_0001" # @param {type:"string"}
activity_type = "Clean & Jerk" # @param {type:"string"}
weight_used = 20 # @param {type:"number"}

exp_id = store.create_entry_interactive(
    entry_type="experiments",
    entity_id=target_subject_id,
    description=activity_type,
    weight_kg=weight_used
)

# @markdown ---
# @markdown ### 📈 Auto-Visualization
# @markdown After upload, the IMU data will be plotted below for quality check.

if exp_id:
    raw_df = store.get_entry_raw_data("experiments", exp_id)

    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    # Dynamically find Accelerometer columns (assuming AX, AY, AZ naming)
    accel_cols = [c for c in raw_df.columns if '_AX' in c or '_AY' in c or '_AZ' in c][:3]

    fig = go.Figure()
    for col in accel_cols:
        fig.add_trace(go.Scatter(y=raw_df[col], name=col, mode='lines'))

    fig.update_layout(
        title=f"Initial Data Check: {exp_id}",
        xaxis_title="Samples",
        yaxis_title="Acceleration ($m/s^2$)",
        template="plotly_dark",
        height=400
    )
    fig.show()

In [ ]:
# @title 📎 Attachment Manager { display-mode: "form" }
# @markdown Upload related files (videos, high-speed clips, or archives) to an experiment.

experiment_to_attach = "experiments_0001" # @param {type:"string"}
file_label = "front_video" # @param ["front_video", "side_video", "raw_zip", "other"]

rel_path = store.upload_related_file_interactive(
    entry_type="experiments",
    entry_id=experiment_to_attach,
    label=file_label
)

# **🧪 Advanced IMU Analysis Plot**

In [ ]:
# @title 🔍 Detailed Signal Analysis { display-mode: "form" }
# @markdown Compare different IMU nodes on the body.

selected_exp = "experiments_0001" # @param {type:"string"}
sensor_node_A = "OrbiX-Node-1" # @param {type:"string"}
sensor_node_B = "OrbiX-Node-9" # @param {type:"string"}

df_analysis = store.get_entry_raw_data("experiments", selected_exp)

import plotly.express as px

# Filter for specific nodes
plot_cols = [c for c in df_analysis.columns if sensor_node_A in c or sensor_node_B in c]
plot_cols = [c for c in plot_cols if "IMU1_AX" in c] # Just showing X-axis for comparison

fig = px.line(df_analysis, y=plot_cols,
              title=f"Comparison: {sensor_node_A} vs {sensor_node_B}",
              labels={"value": "G-Force", "index": "Time (Samples)"},
              template="simple_white")

fig.show()